# RSNA Knee — Audit Kualitas DICOM EXHAUSTIVE (semua ~820rb file)

Versi paralel + resumable dari `rsna_knee_dicom_audit.ipynb`. Beda dari versi sampel:

- **Semua series di `train_series.csv`** diaudit (bukan sampel), setiap instance dapat analisis pixel penuh (bukan dibatasi 25/series)
- **Paralel** (multiprocessing, 4 worker) supaya muat dalam waktu wajar
- **Resumable**: kalau ada file `prior_checkpoint.csv` ter-bundle bersama notebook ini, series yang sudah pernah selesai di run sebelumnya di-skip, hasil baru ditambahkan
- **Wall-clock safety cutoff**: berhenti bersih + simpan checkpoint sebelum kena batas sesi Kaggle, bukan mati mendadak
- `DEBUG_LIMIT_SERIES` di bawah bisa diisi angka kecil untuk uji cepat sebelum run penuh

## 0. Setup

In [ ]:
import os, sys, json, hashlib, traceback, warnings, random, time
from pathlib import Path
from collections import Counter, defaultdict
from multiprocessing import Pool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 160)
RNG = np.random.default_rng(42)
random.seed(42)

T0 = time.time()
def tlog(msg):
    print(f"[t={time.time()-T0:7.1f}s] {msg}", flush=True)

def ensure(pkg, import_name=None):
    import importlib
    name = import_name or pkg
    try:
        importlib.import_module(name); return True
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q {pkg}")
        try:
            importlib.import_module(name); return True
        except Exception as e:
            print(f"[warn] gagal import/install {pkg}: {e}"); return False

HAVE_PYDICOM = ensure("pydicom")
if HAVE_PYDICOM:
    import pydicom

OUT_DIR = Path("audit_outputs")
OUT_DIR.mkdir(exist_ok=True)
ACCENT = "#3B7DD8"
PALETTE = ["#3B7DD8", "#D8763B", "#4CA36B", "#B24C63", "#7A5CC0", "#C9A227", "#4CA3A3", "#8C8C8C"]
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "axes.grid": True,
                      "grid.color": "#e6e6e6", "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold"})
def savefig(fig, name, **kw):
    fig.tight_layout(); fig.savefig(OUT_DIR / name, dpi=140, bbox_inches="tight", **kw)


In [ ]:
kaggle_input = Path("/kaggle/input")
CANDIDATE_ROOTS = [Path("/kaggle/input/rsna-knee-abnormality-detection"),
                    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                    Path("../data"), Path("data")]
if kaggle_input.exists():
    for pattern in ("*/train.csv", "*/test.csv", "*/*/train.csv", "*/*/test.csv"):
        for hit in kaggle_input.glob(pattern):
            CANDIDATE_ROOTS.insert(0, hit.parent)
DATA_DIR = next((p for p in CANDIDATE_ROOTS if p.exists() and ((p/"train.csv").exists() or (p/"test.csv").exists())), None) \
           or next((p for p in CANDIDATE_ROOTS if p.exists()), CANDIDATE_ROOTS[0])
tlog(f"DATA_DIR: {DATA_DIR} | exists: {DATA_DIR.exists()}")

def read_csv_safe(name):
    p = DATA_DIR / name
    if not p.exists():
        print(f"[missing] {name}"); return None
    return pd.read_csv(p)

train = read_csv_safe("train.csv")
train_series = read_csv_safe("train_series.csv")
tlog(f"train: {None if train is None else train.shape} | train_series: {None if train_series is None else train_series.shape}")

LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
LABEL_COLS = [c for c in LABEL_COLS if train is not None and c in train.columns]
gold_studies = set(train.loc[train[LABEL_COLS].notna().any(axis=1), 'StudyInstanceUID']) if train is not None else set()

TRAIN_DIR = DATA_DIR / "train_series"

def list_dcm(study, series):
    d = TRAIN_DIR / study / series
    return sorted(d.glob("*.dcm")) if d.exists() else []


## 1. Cek checkpoint dari run sebelumnya (kalau ada)

In [ ]:
# CATATAN: file lepas di folder push kernel TIDAK ikut ter-upload (sudah diverifikasi -- `kaggle kernels push`
# cuma bawa code_file + kernel-metadata.json). Cara resmi Kaggle utk nempelin checkpoint run sebelumnya
# adalah lewat Kaggle Dataset yang di-attach via `dataset_sources` di kernel-metadata.json -- dataset itu
# akan ter-mount di /kaggle/input/<slug-dataset>/. Auto-discover shallow (1 level) di sini, aman & konsisten
# dengan pola DATA_DIR di atas.
PRIOR_CHECKPOINT_CANDIDATES = [Path("/kaggle/working/prior_checkpoint.csv"), Path("prior_checkpoint.csv")]
if kaggle_input.exists():
    for hit in kaggle_input.glob("*/prior_checkpoint.csv"):
        PRIOR_CHECKPOINT_CANDIDATES.insert(0, hit)
prior_checkpoint_path = next((p for p in PRIOR_CHECKPOINT_CANDIDATES if p.exists()), None)

done_series_keys = set()
FULL_CSV_PATH = OUT_DIR / "audit_per_instance_FULL.csv"
FINAL_COLS = ["StudyInstanceUID","SeriesInstanceUID","SOPInstanceUID","file_path",
              "error_category","severity_score","catatan","Anatomical_Plane",
              "Fluid_Sensitive","Fat_Suppression","blur_score","mean_intensity","std_intensity"]

if prior_checkpoint_path is not None:
    tlog(f"Ketemu checkpoint sebelumnya: {prior_checkpoint_path}")
    prior_df = pd.read_csv(prior_checkpoint_path)
    done_series_keys = set(zip(prior_df["StudyInstanceUID"], prior_df["SeriesInstanceUID"]))
    prior_df[FINAL_COLS].to_csv(FULL_CSV_PATH, index=False)  # jadi titik awal file akumulasi
    tlog(f"Series yang SUDAH selesai dari run sebelumnya: {len(done_series_keys)} series, {len(prior_df)} baris instance")
else:
    tlog("Tidak ada checkpoint sebelumnya -- mulai dari nol.")
    pd.DataFrame(columns=FINAL_COLS).to_csv(FULL_CSV_PATH, index=False)


## 2. Fungsi worker (dijalankan paralel per series)

In [ ]:
REQUIRED_TAGS = ["Rows", "Columns", "BitsAllocated", "PixelSpacing", "Modality"]

def quick_thumb_hash(pixel_array, size=32):
    arr = pixel_array.astype(np.float32)
    ptp = np.ptp(arr)
    norm = np.zeros_like(arr, dtype=np.uint8) if ptp < 1e-9 else ((arr - arr.min()) / ptp * 255).astype(np.uint8)
    h, w = norm.shape[:2]
    row_idx = np.linspace(0, h - 1, min(size, h)).astype(int)
    col_idx = np.linspace(0, w - 1, min(size, w)).astype(int)
    return hashlib.md5(norm[np.ix_(row_idx, col_idx)].tobytes()).hexdigest()

def pixel_quality_metrics(pixel_array):
    arr = pixel_array.astype(np.float64)
    ptp = float(np.ptp(arr))
    is_constant = ptp < 1e-6
    blur_score = None if is_constant else float(ndi.laplace(arr).var())
    return dict(mean=float(arr.mean()), std=float(arr.std()), ptp=ptp,
                p1=float(np.percentile(arr, 1)), p99=float(np.percentile(arr, 99)),
                is_constant=is_constant, blur_score=blur_score)

def process_one_series(args):
    """Worker: proses SEMUA instance dalam satu series. Return dict siap-gabung."""
    study, series, plane, fs, fatsup = args
    d = TRAIN_DIR / study / series
    files = sorted(d.glob("*.dcm")) if d.exists() else []
    rows, errlog, hashes = [], [], []

    if not files:
        return dict(rows=[dict(StudyInstanceUID=study, SeriesInstanceUID=series, SOPInstanceUID=None,
                                file_path=None, error_category="MISSING_FOLDER_FOR_CSV", severity_score=10,
                                catatan="folder series tidak ditemukan", Anatomical_Plane=plane,
                                Fluid_Sensitive=fs, Fat_Suppression=fatsup, blur_score=None,
                                mean_intensity=None, std_intensity=None)],
                    hdr_list=[], errlog=[], study=study, series=series)

    hdr_list = []
    for f in files:
        try:
            ds = pydicom.dcmread(f, force=True)
        except Exception as e:
            errlog.append({"context": f"dcmread {f}", "exception": str(e), "traceback": traceback.format_exc()})
            rows.append(dict(StudyInstanceUID=study, SeriesInstanceUID=series, SOPInstanceUID=f.stem,
                              file_path=str(f), error_category="CORRUPT_FILE", severity_score=10,
                              catatan=f"dcmread gagal: {e}", Anatomical_Plane=plane, Fluid_Sensitive=fs,
                              Fat_Suppression=fatsup, blur_score=None, mean_intensity=None, std_intensity=None))
            continue

        sop = getattr(ds, "SOPInstanceUID", f.stem)
        missing_tags = [t for t in REQUIRED_TAGS if t not in ds]
        rows_, cols_ = getattr(ds, "Rows", None), getattr(ds, "Columns", None)
        ps = getattr(ds, "PixelSpacing", None)
        hdr_list.append(dict(InstanceNumber=getattr(ds, "InstanceNumber", None), Rows=rows_, Columns=cols_,
                              PixelSpacing=float(ps[0]) if ps else None,
                              SliceThickness=getattr(ds, "SliceThickness", None),
                              Orientation=tuple(round(float(x), 2) for x in getattr(ds, "ImageOrientationPatient", [])) or None))

        cats = []
        if missing_tags:
            cats.append("MISSING_TAG")
        if rows_ in (None, 0) or cols_ in (None, 0) or (rows_ and rows_ < 32) or (cols_ and cols_ < 32):
            cats.append("ABNORMAL_DIMENSION")
        elif rows_ and cols_ and (max(rows_, cols_) / max(1, min(rows_, cols_)) > 4):
            cats.append("ABNORMAL_DIMENSION")

        metrics = {}
        try:
            arr = ds.pixel_array
            metrics = pixel_quality_metrics(arr)
            if metrics["is_constant"]:
                cats.append("EMPTY_OR_CONSTANT")
            h = quick_thumb_hash(arr)
            if h in hashes:
                cats.append("DUPLICATE_SLICE")
            hashes.append(h)
        except Exception as e:
            errlog.append({"context": f"pixel_array {f}", "exception": str(e), "traceback": traceback.format_exc()})
            cats.append("CORRUPT_FILE")

        rows.append(dict(StudyInstanceUID=study, SeriesInstanceUID=series, SOPInstanceUID=sop, file_path=str(f),
                          error_category="|".join(cats) if cats else "", severity_score=len(cats),
                          catatan="; ".join(missing_tags) if missing_tags else "",
                          Anatomical_Plane=plane, Fluid_Sensitive=fs, Fat_Suppression=fatsup,
                          blur_score=metrics.get("blur_score"), mean_intensity=metrics.get("mean"),
                          std_intensity=metrics.get("std")))

    return dict(rows=rows, hdr_list=hdr_list, errlog=errlog, study=study, series=series)


## 3. Loop utama (paralel, batch, checkpoint, wall-clock cutoff)

In [ ]:
DEBUG_LIMIT_SERIES = None   # isi angka kecil (mis. 300) untuk uji cepat; None = semua series
N_WORKERS = 4
BATCH_SIZE = 200
SAFETY_CUTOFF_SECONDS = 8 * 3600  # berhenti bersih sebelum sesi Kaggle mentok (~9 jam)
CHECKPOINT_EVERY_BATCH = 1

todo = train_series.sort_values(['StudyInstanceUID', 'SeriesInstanceUID']).reset_index(drop=True)
todo_keys = list(zip(todo['StudyInstanceUID'], todo['SeriesInstanceUID']))
remaining_mask = [k not in done_series_keys for k in todo_keys]
todo = todo[remaining_mask].reset_index(drop=True)
if DEBUG_LIMIT_SERIES:
    todo = todo.head(DEBUG_LIMIT_SERIES)

tlog(f"Total series di train_series.csv: {len(train_series)}")
tlog(f"Sudah selesai dari checkpoint sebelumnya: {len(done_series_keys)}")
tlog(f"AKAN diproses run ini: {len(todo)}")

work_items = list(zip(todo['StudyInstanceUID'], todo['SeriesInstanceUID'], todo['Anatomical_Plane'],
                       todo['Fluid_Sensitive'], todo['Fat_Suppression']))

all_series_consistency = []
all_errlog = []
n_series_done_this_run = 0
n_instances_done_this_run = 0
hit_time_limit = False

if work_items:
    loop_t0 = time.time()
    with Pool(N_WORKERS) as pool:
        for batch_start in range(0, len(work_items), BATCH_SIZE):
            elapsed = time.time() - T0
            if elapsed > SAFETY_CUTOFF_SECONDS:
                tlog(f"BATAS WAKTU AMAN TERCAPAI ({elapsed/3600:.1f} jam) -- berhenti bersih, simpan checkpoint.")
                hit_time_limit = True
                break

            batch = work_items[batch_start: batch_start + BATCH_SIZE]
            results = pool.map(process_one_series, batch)

            batch_rows = []
            for r in results:
                batch_rows.extend(r["rows"])
                all_errlog.extend(r["errlog"])
                n_series_done_this_run += 1
                n_instances_done_this_run += len(r["rows"])
                if r["hdr_list"]:
                    hdr_df_s = pd.DataFrame(r["hdr_list"])
                    n_unique_dim = hdr_df_s[["Rows","Columns"]].drop_duplicates().shape[0]
                    n_unique_spacing = hdr_df_s["PixelSpacing"].nunique(dropna=True)
                    n_unique_thick = hdr_df_s["SliceThickness"].nunique(dropna=True)
                    inst_nums = hdr_df_s["InstanceNumber"].dropna().sort_values().tolist()
                    has_gap = bool((np.diff(inst_nums) > 1).any()) if len(inst_nums) >= 2 else False
                    n_unique_orient = hdr_df_s["Orientation"].dropna().nunique()
                    all_series_consistency.append(dict(
                        StudyInstanceUID=r["study"], SeriesInstanceUID=r["series"], n_instance=len(r["hdr_list"]),
                        n_unique_dim=n_unique_dim, n_unique_spacing=n_unique_spacing, n_unique_thickness=n_unique_thick,
                        has_instance_number_gap=has_gap, n_unique_orientation=n_unique_orient,
                        INCONSISTENT_SERIES=(n_unique_dim > 1 or n_unique_spacing > 1 or n_unique_thick > 1),
                        ORIENTATION_ANOMALY=(n_unique_orient > 1)))

            pd.DataFrame(batch_rows)[FINAL_COLS].to_csv(FULL_CSV_PATH, mode="a", header=False, index=False)

            elapsed = time.time() - T0
            loop_elapsed = time.time() - loop_t0
            n_done_total = len(done_series_keys) + n_series_done_this_run
            pct = n_done_total / len(train_series) * 100
            rate = n_series_done_this_run / max(1e-6, loop_elapsed)  # series/detik run ini
            remaining = len(work_items) - n_series_done_this_run
            eta_min = (remaining / rate / 60) if rate > 0 else float("nan")
            tlog(f"batch {batch_start//BATCH_SIZE + 1}: {n_series_done_this_run}/{len(work_items)} series run ini "
                 f"({n_done_total}/{len(train_series)} total = {pct:.1f}%), {n_instances_done_this_run} instance, "
                 f"{elapsed/60:.1f} menit berjalan, {rate*60:.1f} series/menit, ETA sisa run ini ~{eta_min:.0f} menit")
else:
    tlog("Tidak ada series tersisa -- semua sudah selesai dari checkpoint sebelumnya!")

tlog(f"SELESAI batch loop. {n_series_done_this_run} series baru diproses run ini "
     f"({n_instances_done_this_run} instance). hit_time_limit={hit_time_limit}")
if all_errlog:
    pd.DataFrame(all_errlog).to_csv(OUT_DIR / "full_exception_log.csv", mode="a",
                                     header=not (OUT_DIR / "full_exception_log.csv").exists(), index=False)
    tlog(f"{len(all_errlog)} exception penuh dicatat.")


In [ ]:
n_total_done = len(done_series_keys) + n_series_done_this_run
n_total_series = len(train_series)
is_fully_complete = (n_total_done >= n_total_series) and not hit_time_limit

print(f"\n{'='*60}")
print(f"STATUS: {'SELESAI PENUH' if is_fully_complete else 'BELUM SELESAI -- perlu run lanjutan'}")
print(f"Progress: {n_total_done} / {n_total_series} series ({n_total_done/n_total_series*100:.1f}%)")
print(f"{'='*60}")

if not is_fully_complete:
    print(f"\n>>> UNTUK LANJUTKAN: tarik audit_per_instance_FULL.csv dari output run ini,")
    print(f">>> jadikan 'prior_checkpoint.csv' di push berikutnya, lalu push ulang kernel yang sama.")


## 4. Klasifikasi & ringkasan (berdasarkan progress SAAT INI -- bisa parsial)

Kalau `is_fully_complete = False`, angka di bawah ini adalah ringkasan dari series yang **sudah** diaudit sejauh ini, bukan seluruh dataset. Tetap berguna sebagai progress report.

In [ ]:
instance_df = pd.read_csv(FULL_CSV_PATH)
print("Total baris terkumpul sejauh ini:", instance_df.shape)

PCTL_LOW, PCTL_HIGH = 2, 98
instance_df["error_category"] = instance_df["error_category"].fillna("")
work = instance_df.copy()
work["group"] = work["Anatomical_Plane"].astype(str) + "|FS" + work["Fluid_Sensitive"].astype(str) + "|Fat" + work["Fat_Suppression"].astype(str)

extra_cats = defaultdict(list)
for grp, g in work.groupby("group"):
    for col, lo_cat, hi_cat in [("blur_score", "BLUR", None), ("mean_intensity", "UNDEREXPOSED", "OVEREXPOSED"),
                                  ("std_intensity", "LOW_CONTRAST", None)]:
        if g[col].notna().sum() < 5:
            continue
        lo, hi = np.nanpercentile(g[col], [PCTL_LOW, PCTL_HIGH])
        for idx, v in g[col].items():
            if pd.isna(v):
                continue
            if v < lo:
                extra_cats[idx].append(lo_cat)
            elif hi_cat and v > hi:
                extra_cats[idx].append(hi_cat)

for idx, cats in extra_cats.items():
    prev = instance_df.at[idx, "error_category"]
    instance_df.at[idx, "error_category"] = "|".join([c for c in ([prev] if prev else []) + cats if c])
    instance_df.at[idx, "severity_score"] = instance_df.at[idx, "severity_score"] + len(cats)

if all_series_consistency:
    cons_df = pd.DataFrame(all_series_consistency)
    bad = cons_df[cons_df["INCONSISTENT_SERIES"] | cons_df["ORIENTATION_ANOMALY"]]
    for _, r in bad.iterrows():
        mask = (instance_df["SeriesInstanceUID"] == r["SeriesInstanceUID"]) & (instance_df["StudyInstanceUID"] == r["StudyInstanceUID"])
        extra = (["INCONSISTENT_SERIES"] if r["INCONSISTENT_SERIES"] else []) + (["ORIENTATION_ANOMALY"] if r["ORIENTATION_ANOMALY"] else [])
        instance_df.loc[mask, "error_category"] = instance_df.loc[mask, "error_category"].apply(
            lambda c: "|".join([x for x in ([c] if c else []) + extra if x]))
        instance_df.loc[mask, "severity_score"] += len(extra)

instance_df.loc[instance_df["error_category"] == "", "error_category"] = "OK"
instance_df.to_csv(OUT_DIR / "audit_per_instance_FULL.csv", index=False)
print("Distribusi kategori:")
print(instance_df["error_category"].str.split("|").explode().value_counts())


In [ ]:
def summarize_series(g):
    cats = g["error_category"].str.split("|").explode()
    cats = cats[cats != "OK"]
    dominant = cats.value_counts().idxmax() if len(cats) else "OK"
    return pd.Series({"n_instance_diaudit": len(g), "n_bermasalah": (g["error_category"] != "OK").sum(),
                       "pct_bermasalah": round((g["error_category"] != "OK").mean() * 100, 1),
                       "kategori_dominan": dominant, "severity_total": g["severity_score"].sum()})

per_series_summary = instance_df.groupby(["StudyInstanceUID","SeriesInstanceUID"]).apply(summarize_series).reset_index()
per_series_summary.to_csv(OUT_DIR / "audit_per_series_summary_FULL.csv", index=False)

def flag_study(g):
    pct_bad = g["pct_bermasalah"].mean()
    has_corrupt = g["kategori_dominan"].isin(["CORRUPT_FILE", "EMPTY_OR_CONSTANT"]).any()
    if has_corrupt or pct_bad > 50:
        return "exclude"
    elif pct_bad > 10 or g["severity_total"].max() >= 5:
        return "needs_review"
    return "usable"

per_study_summary = per_series_summary.groupby("StudyInstanceUID").apply(
    lambda g: pd.Series({"n_series_diaudit": len(g), "pct_bermasalah_avg": round(g["pct_bermasalah"].mean(), 1),
                          "flag": flag_study(g), "is_gold_label": g["StudyInstanceUID"].iloc[0] in gold_studies})
).reset_index()
per_study_summary.to_csv(OUT_DIR / "audit_per_study_summary_FULL.csv", index=False)
print("Distribusi flag studi (progress saat ini):")
print(per_study_summary["flag"].value_counts())
print("\nKHUSUS studi gold-label:")
print(per_study_summary[per_study_summary["is_gold_label"]]["flag"].value_counts())


In [ ]:
total_instances = len(instance_df)
n_ok = (instance_df["error_category"] == "OK").sum()
cat_counts = instance_df["error_category"].str.split("|").explode()
cat_counts = cat_counts[cat_counts != "OK"].value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
if len(cat_counts):
    ax.barh(cat_counts.index[::-1], cat_counts.values[::-1], color=ACCENT)
ax.set_xlabel("jumlah instance")
ax.set_title(f"Progress: {n_total_done}/{n_total_series} series ({n_total_done/n_total_series*100:.1f}%) | {total_instances} instance | {n_ok/max(1,total_instances)*100:.1f}% bersih")
savefig(fig, "audit_error_distribution_FULL.png")
plt.show()

TOP_N = 50
top_priority = instance_df[instance_df["error_category"] != "OK"].sort_values("severity_score", ascending=False).head(TOP_N)
top_priority.to_csv(OUT_DIR / "audit_top_priority_FULL.csv", index=False)
print(f"Total instance: {total_instances} | bersih: {n_ok} ({n_ok/max(1,total_instances)*100:.1f}%)")


## Catatan

- Ini progress **kumulatif** lintas semua run (checkpoint di-bundle otomatis kalau ada).
- Kalau status di atas "BELUM SELESAI", jalankan lagi: tarik `audit_per_instance_FULL.csv`, taruh sebagai `prior_checkpoint.csv` di folder kernel ini, push ulang -- series yang sudah selesai otomatis di-skip.
- `SAFETY_CUTOFF_SECONDS` (default 8 jam) memastikan kernel berhenti bersih sebelum kena batas sesi Kaggle, bukan mati mendadak kehilangan progress.